# scib Benchmarking — All Integration Methods

Loads the shared object with all latent representations, runs PCA as unintegrated baseline,
and benchmarks all methods with scib-metrics in one combined table.

In [ ]:
METHODS_PATH = "/vol/disk/ubuntu/master_practicum_cytokines/data/data_for_practicum_integration_methods.h5ad"
OUTPUT_DIR   = "/vol/disk/ubuntu/master_practicum_cytokines/lisa"
SAMPLE_COL   = "library"
LABEL_KEY    = "cell_type_Scanorama"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Input:  {METHODS_PATH}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)

## Step 1 — Load shared object

In [ ]:
adata = sc.read_h5ad(METHODS_PATH)
print(f"Cells: {adata.n_obs:,}   Genes: {adata.n_vars:,}")

print(f"\nEmbeddings: {sorted(adata.obsm.keys())}")
print(f"Label key '{LABEL_KEY}': {adata.obs[LABEL_KEY].nunique()} cell types")
print(adata.obs[LABEL_KEY].value_counts())

## Step 2 — Confirm X_pca is present (unintegrated baseline)

In [ ]:
adata.X = adata.layers["log1p_norm"]
adata.var["highly_variable"] = adata.var["hvg"]
sc.pp.pca(adata, svd_solver="arpack", mask_var="highly_variable")
print(f"X_pca computed: {adata.obsm['X_pca'].shape[1]} components (unintegrated baseline)")

## Step 3 — scib benchmarking

In [ ]:
from scib_metrics.benchmark import Benchmarker

# Auto-detect all latent representations, exclude UMAPs
embedding_keys = sorted([
    k for k in adata.obsm.keys()
    if k.startswith("X_") and "umap" not in k.lower()
])
print(f"Benchmarking embeddings: {embedding_keys}")
print(f"label_key: {LABEL_KEY}")
print(f"batch_key: {SAMPLE_COL}\n")

bm = Benchmarker(
    adata,
    batch_key=SAMPLE_COL,
    label_key=LABEL_KEY,
    embedding_obsm_keys=embedding_keys,
    n_jobs=-1,
)
bm.benchmark()
results = bm.get_results(min_max_scale=False)

numeric_cols = [c for c in results.columns if c != "Metric Type"]
table = results[numeric_cols].round(3)
table.index.name = "Metric"

print("\nscib results:")
display(table)
table.to_csv(os.path.join(OUTPUT_DIR, "scib_all_methods.csv"))
print(f"\nSaved to: {OUTPUT_DIR}/scib_all_methods.csv")